_Updated date: November 20, 2025_

# 🎓 Databricks Workshop: Data & Analytics
**For Business Intelligence & Data Analytics Professionals**

---

## 👥 Welcome, BI & Analytics Team!

This workshop is specifically designed for **Business Intelligence and Data Analytics professionals** who are:
- 📊 Transitioning from **SAS to Databricks** for analytics workloads
- 💼 Building **production-ready data pipelines** for business intelligence
- 📈 Creating **executive dashboards** and analytical reports
- 🔄 Modernizing **legacy analytics** infrastructure
- ⚡ Improving **performance and scalability** of analytics workloads

**Your Role**: As BI analysts, you'll learn how to leverage Databricks to build scalable, maintainable analytics solutions that replace traditional SAS workflows.

---

## 📚 Workshop Objectives

By the end of this workshop, you will be able to:

1. ✅ Build **Medallion Architecture** pipelines for enterprise data
2. ✅ **Translate SAS code** to Databricks SQL and PySpark
3. ✅ Create **aggregated analytics** and business metrics
4. ✅ Perform **data quality audits** and validation checks
5. ✅ Build **Gold layer analytics** for business insights and reporting
6. ✅ Apply **best practices** for production pipelines (caching, checkpointing, deterministic execution)
7. ✅ Optimize **query performance** using Databricks features

---

## 💼 Business Use Case: Enterprise Analytics

This workshop uses a **healthcare payer dataset** as an example, but the concepts apply to any industry:

- 📊 **Customer Analytics**: Understand customer demographics and behavior
- 💰 **Revenue Analysis**: Track financial performance and trends
- 🎯 **Segmentation**: Group customers by attributes for targeted strategies
- 📈 **Performance Metrics**: Calculate KPIs for executive reporting
- ✅ **Data Quality**: Ensure accuracy and completeness for decision-making

### 🗂️ Dataset Overview

We'll work with a **payer dataset** containing:
- **Members**: Customer demographics and attributes
- **Claims**: Transaction records with details
- **Diagnoses**: Classification codes for categorization
- **Providers**: Service provider information
- **Procedures**: Service details and associated costs

**Note**: While this uses healthcare data, the techniques apply to any domain (retail, finance, manufacturing, etc.)

---



# Databricks Medallion Architecture for Business Intelligence


## Business Analytics & Data Modeling Concepts

### 📊 Understanding the Dataset

For this workshop, we'll use a **healthcare payer dataset** as our example. The concepts you'll learn apply universally to any business domain.

**Key Concepts:**
1. **Customer Data** → Demographics, attributes, segments
2. **Transaction Data** → Claims, purchases, interactions
3. **Reference Data** → Categories, codes, mappings
4. **Provider/Vendor Data** → Service providers, suppliers
5. **Metrics & KPIs** → Calculated business measures

### 🏗️ Data Architecture Patterns

**Medallion Architecture** is an industry-standard approach for organizing data:

- **Bronze Layer** (Raw): Data ingested as-is from source systems
- **Silver Layer** (Cleansed): Validated, deduplicated, conformed data
- **Gold Layer** (Analytics)**: Business-level aggregates and metrics

### 📊 Data Model Overview

For our example dataset, key tables include:
- **Members**: Customer/member demographics and attributes
- **Claims**: Transaction records with financial details
- **Diagnoses**: Classification codes for categorization
- **Providers**: Service provider information
- **Procedures**: Service details and costs

<div style="display: flex; justify-content: space-between;">
  <img src="https://user-gen-media-assets.s3.amazonaws.com/gpt4o_images/5c87faea-3e60-4f71-826d-42d04f6cdc0b.png" alt="Dimensional Model" width="400" height="350">
  <img src="https://user-gen-media-assets.s3.amazonaws.com/gpt4o_images/6826c275-d462-4c07-a978-43fe9c40f3ed.png" alt="Data Vault" width="400" height="350">
</div>

**Resources:**
- [Implementing Dimensional Modeling on Databricks](https://www.databricks.com/blog/implementing-dimensional-data-warehouse-databricks-sql-part-1)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)







# 🔄 Why Migrate from SAS to Databricks?

### Common Challenges with SAS for BI Analytics

Organizations using SAS for business intelligence and analytics face several limitations:

1. **🐌 Performance Bottlenecks**
   - Single-server processing limits scalability
   - Large datasets (millions or billions of records) cause memory issues
   - Batch processing takes hours or overnight
   - Difficult to handle real-time or near-real-time analytics

2. **💰 Cost Concerns**
   - Expensive annual licensing fees
   - Additional costs for SAS/ACCESS, SAS Enterprise Guide, SAS Visual Analytics
   - Hardware upgrades needed for growing data volumes
   - Fixed costs regardless of actual usage

3. **🔧 Development Complexity**
   - Multiple PROC steps required for simple operations
   - Limited modern SQL features (no EXPLODE, limited window functions)
   - SAS macros difficult to maintain and debug
   - Separate tools needed for visualization and dashboards
   - Steep learning curve for new team members

4. **☁️ Cloud Migration Challenges**
   - Legacy on-premises architecture
   - Difficult integration with cloud data lakes
   - Limited real-time analytics capabilities
   - Manual scaling and infrastructure management

### Databricks Advantages for BI Teams

| **Capability** | **Impact** |
|----------------|------------|
| **Distributed Processing** | Handle billions of records with sub-second queries |
| **Modern SQL** | Window functions, CTEs, EXPLODE - cleaner, more maintainable code |
| **Unified Platform** | SQL, Python, R, dashboards, ML - all in one place |
| **Delta Lake** | ACID transactions + time travel for audit and compliance |
| **Unity Catalog** | Centralized governance with fine-grained access control |
| **Real-time Analytics** | Streaming + batch unified for immediate insights |
| **Cost Efficiency** | Pay-per-use vs. fixed licensing, auto-scaling |
| **Collaboration** | Shared notebooks, version control, team workspaces |


---


# SETUP

Just run next couple of cells for setup!

In [0]:
dbutils.widgets.text("catalog", "my_catalog", "Catalog")
dbutils.widgets.text("bronze_db", "payer_bronze", "Bronze DB")
dbutils.widgets.text("silver_db", "payer_silver", "Silver DB")
dbutils.widgets.text("gold_db", "payer_gold", "Gold DB")

catalog = dbutils.widgets.get("catalog")
bronze_db = dbutils.widgets.get("bronze_db")
silver_db = dbutils.widgets.get("silver_db")
gold_db = dbutils.widgets.get("gold_db")

path = f"/Volumes/{catalog}/{bronze_db}/payer/files/"

print(f"Catalog: {catalog}")
print(f"Bronze DB: {bronze_db}")
print(f"Silver DB: {silver_db}")
print(f"Gold DB: {gold_db}")
print(f"Path: {path}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_db}.payer")

# Create the volume and folders
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/claims")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/members")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/providers")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/downloads")

In [0]:
import requests
import zipfile
import io
import os
import shutil

# Define the URL of the ZIP file
url = "https://github.com/bigdatavik/databricksfirststeps/blob/6b225621c3c010a2734ab604efd79c15ec6c71b8/data/Payor_Archive.zip?raw=true"

# Download the ZIP file
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# Define the base path
base_path = f"/Volumes/{catalog}/{bronze_db}/payer/downloads" 

# Extract the ZIP file to the base path
zip_file.extractall(base_path)

# Define the paths
paths = {
    "claims.csv": f"{base_path}/claims",
    "diagnoses.csv": f"{base_path}/diagnosis",
    "procedures.csv": f"{base_path}/procedures",
    "member.csv": f"{base_path}/members",
    "providers.csv": f"{base_path}/providers"
}

# Create the destination directories if they do not exist
for dest_path in paths.values():
    os.makedirs(dest_path, exist_ok=True)

# Move the files to the respective directories
for file_name, dest_path in paths.items():
    source_file = f"{base_path}/{file_name}"
    if os.path.exists(source_file):
        os.rename(source_file, f"{dest_path}/{file_name}")


# Copy the files to the specified directories and print the paths
shutil.copy(f"{base_path}/claims/claims.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")

shutil.copy(f"{base_path}/diagnosis/diagnoses.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")

shutil.copy(f"{base_path}/procedures/procedures.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")

shutil.copy(f"{base_path}/members/member.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")

shutil.copy(f"{base_path}/providers/providers.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")



# 🚀 Let's Build Your First Data Pipeline!

---

## Workshop Roadmap

```
📥 Bronze Layer    →    🔧 Silver Layer    →    ⭐ Gold Layer    →    📊 Analytics
   (Raw Data)          (Cleaned Data)        (Business Tables)      (Insights)
```

In the following sections, we'll build a complete data pipeline following the **Medallion Architecture**:

1. **Bronze Layer**: Ingest raw CSV files into Delta tables
2. **Silver Layer**: Clean, deduplicate, and transform data
3. **Gold Layer**: Create enriched analytics tables
4. **Analytics**: Generate insights and visualizations

Let's get started! 🎉

# 📥 Bronze/Silver Layers – Streamlined Data Preparation

---

## Overview: Simplified Bronze & Silver

For analytics, we'll **streamline** Bronze and Silver layers to quickly get to Gold layer insights:

### Bronze Layer (Raw Data Landing)
- 📂 Ingest encounter data "as-is" using `COPY INTO`
- 💾 Store in Delta Lake for audit trails
- ⏱️ Maintain full history for compliance

### Silver Layer (Clean & Validate)
- 🧹 Remove duplicates and validate data quality
- 🔄 Map ICD-10 codes to HCC categories
- ✅ Apply business rules for CMS submission eligibility

> **💡 Focus**: We'll execute Bronze/Silver steps efficiently so we can spend more time on **Gold layer analytics** that drive business value!

---



## Step 1: Verify Source Files

Let's first check that our source files are available:

In [0]:
%sql
LIST '/Volumes/my_catalog/payer_bronze/payer/files/claims/'

## Step 2: Load Data with COPY INTO

### 📖 Understanding COPY INTO

`COPY INTO` is Databricks' recommended command for loading data from cloud storage into Delta tables.

**Key Benefits:**
- ✅ **Idempotent**: Safely re-run without duplicating data
- ✅ **Incremental**: Only loads new files automatically
- ✅ **Schema Evolution**: Can merge new columns with `mergeSchema` option
- ✅ **Atomic**: Either succeeds completely or rolls back

**Syntax:**
```sql
COPY INTO <table_name>
FROM '<source_path>'
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS('mergeSchema' = 'true')
```

📚 **Learn More:**
- [COPY INTO Documentation](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)
- [COPY INTO Examples](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/copy-into/)


### Loading Data with SQL

In [0]:
%sql
-- Load Claims Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.claims_raw;
COPY INTO payer_bronze.claims_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/claims/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'true');

-- NOTE: 'force = true' is used here for demo purposes only to reload all files every time. In production, omit this option so COPY INTO only processes new data files.


-- Load Diagnosis Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.diagnosis_raw;
COPY INTO payer_bronze.diagnosis_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/diagnosis/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Members Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.members_raw;
COPY INTO payer_bronze.members_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/members/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Procedures Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.procedures_raw;
COPY INTO payer_bronze.procedures_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/procedures/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Providers Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.providers_raw;
COPY INTO payer_bronze.providers_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/providers/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


### 🐍 Alternative: Loading Data with PySpark

While SQL is great for batch loading, PySpark gives you more programmatic control. Here's how to load the same data using PySpark:

In [0]:
# Example: Load data using PySpark
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# Option 1: Let Spark infer the schema
claims_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/my_catalog/payer_bronze/payer/files/claims/")

# Display first 10 rows
display(claims_df.limit(10))

# Show schema
print("Claims Schema:")
claims_df.printSchema()

# Get row count
print(f"\nTotal rows loaded: {claims_df.count()}")

# Write to Delta table (this creates or replaces the table)
# claims_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("payer_bronze.claims_raw_pyspark")


## Silver Layer – Transformation


## Step 1: Transform Bronze to Silver (SQL)

Let's clean and transform our Bronze tables. We'll demonstrate with multiple examples using both **SQL** and **PySpark**.

In [0]:
%sql
-- Create silver schema
CREATE SCHEMA IF NOT EXISTS payer_silver;


-- Members: select relevant fields, cast types, remove duplicates
CREATE OR REPLACE TABLE payer_silver.members AS
SELECT
  DISTINCT CAST(member_id AS STRING) AS member_id,
  TRIM(first_name) AS first_name,
  TRIM(last_name) AS last_name,
  CAST(birth_date AS DATE) AS birth_date,
  gender,
  plan_id,
  CAST(effective_date AS DATE) AS effective_date
FROM payer_bronze.members_raw
WHERE member_id IS NOT NULL;


-- Claims: remove duplicates, prepare data
CREATE OR REPLACE TABLE payer_silver.claims AS
SELECT
  DISTINCT claim_id,
  member_id,
  provider_id,
  CAST(claim_date AS DATE) AS claim_date,
  ROUND(total_charge, 2) AS total_charge,
  LOWER(claim_status) AS claim_status
FROM payer_bronze.claims_raw
WHERE claim_id IS NOT NULL AND total_charge > 0;


-- Providers: deduplicate
CREATE OR REPLACE TABLE payer_silver.providers AS
SELECT
  DISTINCT provider_id,
  npi,
  provider_name,
  specialty,
  address,
  city,
  state
FROM payer_bronze.providers_raw
WHERE provider_id IS NOT NULL;


## Step 2: Transform with PySpark

Now let's see how to do the same transformations using PySpark. This approach is more flexible for complex business logic.

### Example: Transform Procedures Table with PySpark


In [0]:
from pyspark.sql.functions import col, trim, upper, round as spark_round, when, regexp_replace

# Read from Bronze
procedures_bronze = spark.table("payer_bronze.procedures_raw")

# Clean and cast the amount column
procedures_bronze_clean = procedures_bronze.withColumn(
    "amount_clean",
    regexp_replace(col("amount"), "[^0-9.]", "").cast("double")
)

# Apply transformations
procedures_silver = procedures_bronze_clean \
    .dropDuplicates(['claim_id', 'procedure_code']) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("amount_clean") > 0) \
    .select(
        col("claim_id"),
        upper(trim(col("procedure_code"))).alias("procedure_code"),
        trim(col("procedure_desc")).alias("procedure_desc"),
        spark_round(col("amount_clean"), 2).alias("amount"),
        when(col("amount_clean") < 100, "Low")
        .when(col("amount_clean") < 500, "Medium")
        .when(col("amount_clean") < 1000, "High")
        .otherwise("Very High").alias("cost_category")
    )

# Show sample data
print("Transformed Procedures (first 10 rows):")
display(procedures_silver.limit(10))

# Show statistics
print("\nCost Category Distribution:")
display(procedures_silver.groupBy("cost_category").count().orderBy("cost_category"))

# Write to Silver table
procedures_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("payer_silver.procedures")


# 🤖 Using Databricks AI Assistant

---

Databricks AI Assistant can help you write code, understand data, and troubleshoot issues!

### How to Use AI Assistant:
1. Click the AI Assistant icon
2. Ask questions in natural language
3. Get code suggestions and explanations

### Example Prompts to Try:
- "How do I calculate the total claims by specialty?"
- "Show me how to create a window function for running totals"
- "What does spark.table() command do?"
- "Help me debug this PySpark error"

---


In [0]:
%sql
SELECT
    p.specialty,
    SUM(c.total_charge) AS total_claims
FROM payer_silver.claims c
JOIN payer_silver.providers p
  ON c.provider_id = p.provider_id
GROUP BY p.specialty
ORDER BY total_claims DESC;


## 🎯 YOUR TURN! (3 mins)
Ask Databricks Assistant: "How do I calculate the total claims by specialty in SQL?"

In [0]:
%sql
-- Solution
SELECT
    claim_status AS specialty,
    SUM(total_charge) AS total_claims
FROM my_catalog.payer_silver.claims
GROUP BY claim_status

# ⭐ Gold Layer – Business Analytics & Insights

---

## What is the Gold Layer?

The **Gold Layer** is where we deliver **business value** for stakeholders. Here we create analytics tables that directly support:

- 📊 **Customer Analytics**: Customer segmentation, demographics, and behavior patterns
- 💰 **Revenue Analysis**: Financial performance, trends, and forecasting
- 📈 **Performance Metrics**: KPIs and business measures for executives
- ✅ **Data Quality Audits**: Validation metrics for data governance
- 🎯 **Segmentation**: Grouping customers/products by business-relevant attributes
- 📉 **Trend Analysis**: Time-series analytics and comparative performance
- 🏆 **Top Performers**: Ranking and leaderboard analyses

> **🎯 Business Value**: Each Gold table directly answers a business question that drives decision-making!

---

## Example 1: Customer Aggregation & Metrics Calculation

### 🎯 Business Goal
Calculate customer-level aggregated metrics by joining multiple tables and applying business logic.

This example demonstrates:
- Multi-table joins
- Aggregations with grouping
- Calculated fields based on business rules
- Window functions for rankings

### 📊 SAS vs. Databricks Comparison

Let's compare how this is traditionally done in **SAS** vs. **Databricks SQL** and **PySpark**:

---

#### Traditional SAS Approach

```sas
/* SAS: HCC Risk Score Calculation */
/* Step 1: Create HCC reference table */
DATA work.hcc_reference;
    INPUT icd10_code $ 1-10 diagnosis_desc $ 12-50 hcc_category hcc_coefficient;
    DATALINES;
E11.9      Type 2 Diabetes                    19  0.318
I50.9      Heart Failure                      85  0.368
I10        Hypertension                       0   0.000
J44.9      COPD                              111  0.328
N18.3      CKD Stage 3                       138  0.237
;
RUN;

/* Step 2: Merge claims with diagnoses and HCC mapping */
PROC SQL;
    CREATE TABLE work.diagnosis_with_hcc AS
    SELECT 
        d.claim_id,
        d.diagnosis_code,
        d.diagnosis_desc,
        h.hcc_category,
        h.hcc_coefficient,
        CASE 
            WHEN h.hcc_category IS NOT NULL AND h.hcc_category > 0 
            THEN 1 ELSE 0 
        END AS is_hcc
    FROM work.diagnosis_raw AS d
    LEFT JOIN work.hcc_reference AS h
        ON UPCASE(STRIP(d.diagnosis_code)) = UPCASE(STRIP(h.icd10_code));
QUIT;

/* Step 3: Calculate member-level risk scores */
PROC SQL;
    CREATE TABLE work.member_hccs AS
    SELECT DISTINCT
        c.member_id,
        m.first_name,
        m.last_name,
        m.birth_date,
        m.gender,
        m.plan_id,
        dh.hcc_category,
        dh.hcc_coefficient,
        dh.diagnosis_code,
        YEAR(TODAY()) - YEAR(m.birth_date) AS age
    FROM work.claims AS c
    INNER JOIN work.members AS m 
        ON c.member_id = m.member_id
    INNER JOIN work.diagnosis_with_hcc AS dh 
        ON c.claim_id = dh.claim_id
    WHERE dh.is_hcc = 1;
QUIT;

/* Step 4: Aggregate and calculate final risk scores */
PROC SQL;
    CREATE TABLE work.member_risk_scores AS
    SELECT 
        member_id,
        first_name,
        last_name,
        birth_date,
        gender,
        plan_id,
        age,
        CASE 
            WHEN age < 65 THEN 0.350
            WHEN age BETWEEN 65 AND 69 THEN 0.450
            WHEN age BETWEEN 70 AND 74 THEN 0.550
            WHEN age BETWEEN 75 AND 79 THEN 0.650
            ELSE 0.750
        END AS demographic_score,
        SUM(hcc_coefficient) AS hcc_score,
        COUNT(DISTINCT hcc_category) AS hcc_count,
        CALCULATED demographic_score + CALCULATED hcc_score AS total_risk_score,
        ROUND((CALCULATED demographic_score + CALCULATED hcc_score) * 10000, 0.01) 
            AS projected_annual_payment
    FROM work.member_hccs
    GROUP BY member_id, first_name, last_name, birth_date, gender, plan_id, age
    ORDER BY total_risk_score DESC;
QUIT;
```

**SAS Challenges:**
- ❌ Multiple PROC SQL steps required
- ❌ Intermediate tables clutter WORK library
- ❌ Limited scalability with large datasets
- ❌ No automatic optimization or parallelization
- ❌ Complex syntax for array aggregations (HCC categories list)

---

Now let's see how **Databricks SQL** simplifies this!

In [0]:
%sql
-- Create gold schema
CREATE SCHEMA IF NOT EXISTS payer_gold;

-- Step 1a: Create HCC Mapping Reference Table (simulated for demo)
CREATE OR REPLACE TABLE payer_gold.hcc_reference (
  icd10_code STRING,
  diagnosis_desc STRING,
  hcc_category INT,
  hcc_coefficient DOUBLE
);

INSERT INTO payer_gold.hcc_reference VALUES
('E11.9', 'Type 2 Diabetes', 19, 0.318),
('I50.9', 'Heart Failure', 85, 0.368),
('I10', 'Hypertension', 0, 0.000),
('J44.9', 'COPD', 111, 0.328),
('N18.3', 'CKD Stage 3', 138, 0.237),
('F32.9', 'Depression', 59, 0.309),
('E78.5', 'Hyperlipidemia', 0, 0.000),
('I25.10', 'CAD', 88, 0.184);

-- Step 1b: Join Diagnoses to HCC Categories
CREATE OR REPLACE TABLE payer_gold.diagnosis_with_hcc AS
SELECT
  d.claim_id,
  d.diagnosis_code,
  d.diagnosis_desc,
  h.hcc_category,
  h.hcc_coefficient,
  CASE WHEN h.hcc_category IS NOT NULL AND h.hcc_category > 0 THEN 1 ELSE 0 END as is_hcc
FROM payer_bronze.diagnosis_raw d
LEFT JOIN payer_gold.hcc_reference h 
  ON UPPER(TRIM(d.diagnosis_code)) = UPPER(TRIM(h.icd10_code));

-- Step 1c: Calculate Member-Level HCC Risk Scores
CREATE OR REPLACE TABLE payer_gold.member_risk_scores AS
WITH member_hccs AS (
  SELECT DISTINCT
    c.member_id,
    m.first_name,
    m.last_name,
    m.birth_date,
    m.gender,
    m.plan_id,
    dh.hcc_category,
    dh.hcc_coefficient,
    dh.diagnosis_code,
    YEAR(CURRENT_DATE()) - YEAR(m.birth_date) as age
  FROM payer_silver.claims c
  INNER JOIN payer_silver.members m ON c.member_id = m.member_id
  INNER JOIN payer_gold.diagnosis_with_hcc dh ON c.claim_id = dh.claim_id
  WHERE dh.is_hcc = 1
),
member_scores AS (
  SELECT
    member_id,
    first_name,
    last_name,
    birth_date,
    gender,
    plan_id,
    age,
    CASE 
      WHEN age < 65 THEN 0.350
      WHEN age BETWEEN 65 AND 69 THEN 0.450
      WHEN age BETWEEN 70 AND 74 THEN 0.550
      WHEN age BETWEEN 75 AND 79 THEN 0.650
      ELSE 0.750
    END as demographic_score,
    SUM(hcc_coefficient) as hcc_score,
    COUNT(DISTINCT hcc_category) as hcc_count,
    COLLECT_SET(hcc_category) as hcc_categories,
    COLLECT_SET(diagnosis_code) as diagnosis_codes
  FROM member_hccs
  GROUP BY member_id, first_name, last_name, birth_date, gender, plan_id, age
)
SELECT
  *,
  demographic_score + hcc_score as total_risk_score,
  ROUND((demographic_score + hcc_score) * 10000, 2) as projected_annual_payment
FROM member_scores
ORDER BY total_risk_score DESC;

### 🚀 Databricks SQL Advantages

**Databricks SQL Benefits:**
- ✅ **Single SQL Statement**: All logic in one CREATE TABLE AS with CTEs
- ✅ **Advanced Functions**: COLLECT_SET() for array aggregation (no SAS equivalent)
- ✅ **Automatic Optimization**: Query engine optimizes joins and aggregations
- ✅ **Scalability**: Distributed processing handles billions of rows
- ✅ **Unity Catalog**: Built-in governance, lineage tracking, and access control
- ✅ **Delta Lake**: ACID transactions, time travel, schema evolution
- ✅ **Real-time Refresh**: Can be scheduled or triggered automatically

### 💡 PySpark Alternative

For complex business logic or programmatic control, use PySpark:


In [0]:
# PySpark Example: HCC Risk Score Calculation
from pyspark.sql.functions import (
    col, year, current_date, sum as _sum, count, countDistinct,
    collect_set, when, round as spark_round, lit
)

# Read tables
claims = spark.table("payer_silver.claims")
members = spark.table("payer_silver.members")
diagnosis_with_hcc = spark.table("payer_gold.diagnosis_with_hcc")

# Join and calculate member HCCs
member_hccs = claims \
    .join(members, "member_id") \
    .join(diagnosis_with_hcc, "claim_id") \
    .filter(col("is_hcc") == 1) \
    .withColumn("age", year(current_date()) - year(col("birth_date"))) \
    .select(
        "member_id", "first_name", "last_name", "birth_date", "gender", 
        "plan_id", "age", "hcc_category", "hcc_coefficient", "diagnosis_code"
    ) \
    .distinct()

# Calculate demographic scores and aggregate HCC scores
member_risk_scores_pyspark = member_hccs \
    .withColumn("demographic_score",
        when(col("age") < 65, 0.350)
        .when(col("age") <= 69, 0.450)
        .when(col("age") <= 74, 0.550)
        .when(col("age") <= 79, 0.650)
        .otherwise(0.750)
    ) \
    .groupBy("member_id", "first_name", "last_name", "birth_date", 
             "gender", "plan_id", "age", "demographic_score") \
    .agg(
        _sum("hcc_coefficient").alias("hcc_score"),
        countDistinct("hcc_category").alias("hcc_count"),
        collect_set("hcc_category").alias("hcc_categories"),
        collect_set("diagnosis_code").alias("diagnosis_codes")
    ) \
    .withColumn("total_risk_score", col("demographic_score") + col("hcc_score")) \
    .withColumn("projected_annual_payment", 
                spark_round(col("total_risk_score") * 10000, 2)) \
    .orderBy(col("total_risk_score").desc())

# Display results
print("📊 PySpark Risk Score Calculation - Top 10 Members:")
display(member_risk_scores_pyspark.limit(10))

# Optionally write to table
# member_risk_scores_pyspark.write.format("delta").mode("overwrite") \
#     .saveAsTable("payer_gold.member_risk_scores_pyspark")


In [0]:
%sql
SELECT 
  member_id,
  CONCAT(first_name, ' ', last_name) AS member_name,
  age,
  gender,
  hcc_count,
  ROUND(demographic_score, 3) AS demo_score,
  ROUND(hcc_score, 3) AS hcc_score,
  ROUND(total_risk_score, 3) AS risk_score,
  projected_annual_payment
FROM payer_gold.member_risk_scores
ORDER BY total_risk_score DESC
LIMIT 20;

## 🎯 YOUR TURN! Risk Score (3 mins)
Ask Databricks Assistant: "Calculate top 20 members by risk score in SQL"

In [0]:
%sql
-- Solution
-- View top 20 members by risk score
SELECT 
  member_id,
  CONCAT(first_name, ' ', last_name) as member_name,
  age,
  gender,
  hcc_count,
  ROUND(demographic_score, 3) as demo_score,
  ROUND(hcc_score, 3) as hcc_score,
  ROUND(total_risk_score, 3) as risk_score,
  projected_annual_payment
FROM payer_gold.member_risk_scores
ORDER BY total_risk_score DESC
LIMIT 20;



## Example 2: Revenue Forecast & Impact Analysis

### 💰 Business Goal
Project total CMS revenue based on risk scores to support financial planning.

**Key Metrics:**
- Total member population
- Average risk score
- Projected annual revenue
- Revenue by plan and risk tier


### 📊 SAS vs. Databricks SQL Comparison

#### Traditional SAS Approach

```sas
/* SAS: Revenue Forecast by Plan */
PROC SQL;
    CREATE TABLE work.revenue_forecast AS
    SELECT 
        plan_id,
        COUNT(DISTINCT member_id) AS total_members,
        ROUND(AVG(total_risk_score), 0.001) AS avg_risk_score,
        ROUND(MIN(total_risk_score), 0.001) AS min_risk_score,
        ROUND(MAX(total_risk_score), 0.001) AS max_risk_score,
        SUM(projected_annual_payment) AS total_projected_revenue,
        ROUND(AVG(projected_annual_payment), 0.01) AS avg_payment_per_member,
        SUM(CASE WHEN total_risk_score >= 1.5 THEN 1 ELSE 0 END) AS high_risk_members,
        SUM(CASE WHEN total_risk_score < 1.0 THEN 1 ELSE 0 END) AS low_risk_members
    FROM work.member_risk_scores
    GROUP BY plan_id
    ORDER BY total_projected_revenue DESC;
QUIT;

/* Export to Excel for reporting */
PROC EXPORT DATA=work.revenue_forecast
    OUTFILE='/path/to/revenue_forecast.xlsx'
    DBMS=XLSX REPLACE;
RUN;

/* Generate summary report */
PROC PRINT DATA=work.revenue_forecast;
    TITLE 'Revenue Forecast by Plan';
RUN;
```

**SAS Limitations:**
- ❌ Manual export steps for reporting
- ❌ No real-time dashboard integration
- ❌ Limited to single-server processing
- ❌ Requires additional tools for visualization
- ❌ Static reports need manual refresh

#### Databricks SQL Approach


In [0]:
%sql
CREATE OR REPLACE TABLE payer_gold.revenue_forecast AS
SELECT
  plan_id,
  COUNT(DISTINCT member_id) AS total_members,
  ROUND(AVG(total_risk_score), 3) AS avg_risk_score,
  ROUND(MIN(total_risk_score), 3) AS min_risk_score,
  ROUND(MAX(total_risk_score), 3) AS max_risk_score,
  SUM(projected_annual_payment) AS total_projected_revenue,
  ROUND(AVG(projected_annual_payment), 2) AS avg_payment_per_member,
  SUM(CASE WHEN total_risk_score >= 1.5 THEN 1 ELSE 0 END) AS high_risk_members,
  SUM(CASE WHEN total_risk_score < 1.0 THEN 1 ELSE 0 END) AS low_risk_members
FROM payer_gold.member_risk_scores
GROUP BY plan_id
ORDER BY total_projected_revenue DESC;

In [0]:
%sql
select * from payer_gold.revenue_forecast


## 🎯 YOUR TURN! Revenue Forecast (5 mins)

### 📋 Business Context for Humana BI Team
As a BI analyst at Humana, you're often asked to create **revenue forecasts by plan** for financial planning. This analysis helps:
- 💰 Project CMS capitation payments
- 📊 Identify which plans drive the most revenue
- 🎯 Target high-risk members for care management programs
- 📈 Support strategic planning and budgeting

### 🔄 Your Task: Convert SAS to Databricks SQL

Below is the **traditional SAS code** used for this analysis. Your job is to convert it to **Databricks SQL**.

**SAS Code (Traditional Approach):**
```sas
PROC SQL;
    CREATE TABLE work.revenue_forecast AS
    SELECT 
        plan_id,
        COUNT(DISTINCT member_id) AS total_members,
        ROUND(AVG(total_risk_score), 0.001) AS avg_risk_score,
        ROUND(MIN(total_risk_score), 0.001) AS min_risk_score,
        ROUND(MAX(total_risk_score), 0.001) AS max_risk_score,
        SUM(projected_annual_payment) AS total_projected_revenue,
        ROUND(AVG(projected_annual_payment), 0.01) AS avg_payment_per_member,
        SUM(CASE WHEN total_risk_score >= 1.5 THEN 1 ELSE 0 END) AS high_risk_members,
        SUM(CASE WHEN total_risk_score < 1.0 THEN 1 ELSE 0 END) AS low_risk_members
    FROM work.member_risk_scores
    GROUP BY plan_id
    ORDER BY total_projected_revenue DESC;
QUIT;
```

### ✍️ Instructions:
1. **Convert the SAS code** to Databricks SQL
2. Create a table called **`payer_gold.revenue_forecast`**
3. Source data from **`payer_gold.member_risk_scores`**
4. **Hint**: Databricks SQL syntax is very similar to SAS PROC SQL, but with some differences:
   - Use `CREATE OR REPLACE TABLE` instead of `CREATE TABLE`
   - `ROUND()` function works similarly but may have slightly different syntax
   - Table names use three-part naming: `catalog.schema.table`

### 💡 Pro Tips:
- Use **Databricks AI Assistant** 
- Try asking: *"Convert this SAS PROC SQL code to Databricks SQL"*
- Test your query with `SELECT * FROM payer_gold.revenue_forecast LIMIT 10` after creation

### 📝 Write Your SQL Below (in the next cell)


In [0]:
%sql
-- Solution
-- Revenue Forecast by Plan
CREATE OR REPLACE TABLE payer_gold.revenue_forecast AS
SELECT
  plan_id,
  COUNT(DISTINCT member_id) as total_members,
  ROUND(AVG(total_risk_score), 3) as avg_risk_score,
  ROUND(MIN(total_risk_score), 3) as min_risk_score,
  ROUND(MAX(total_risk_score), 3) as max_risk_score,
  SUM(projected_annual_payment) as total_projected_revenue,
  ROUND(AVG(projected_annual_payment), 2) as avg_payment_per_member,
  SUM(CASE WHEN total_risk_score >= 1.5 THEN 1 ELSE 0 END) as high_risk_members,
  SUM(CASE WHEN total_risk_score < 1.0 THEN 1 ELSE 0 END) as low_risk_members
FROM payer_gold.member_risk_scores
GROUP BY plan_id
ORDER BY total_projected_revenue DESC;


### 🐍 BONUS: PySpark Alternative

For BI analysts comfortable with Python, here's the PySpark equivalent:


In [0]:
# 🐍 PySpark Solution: Revenue Forecast
from pyspark.sql.functions import (
    col, countDistinct, avg, min as _min, max as _max, 
    sum as _sum, round as spark_round, when
)

# Read member risk scores
member_scores = spark.table("payer_gold.member_risk_scores")

# Build the aggregation
revenue_forecast_pyspark = member_scores.groupBy("plan_id").agg(
    countDistinct("member_id").alias("total_members"),
    spark_round(avg("total_risk_score"), 3).alias("avg_risk_score"),
    spark_round(_min("total_risk_score"), 3).alias("min_risk_score"),
    spark_round(_max("total_risk_score"), 3).alias("max_risk_score"),
    _sum("projected_annual_payment").alias("total_projected_revenue"),
    spark_round(avg("projected_annual_payment"), 2).alias("avg_payment_per_member"),
    _sum(when(col("total_risk_score") >= 1.5, 1).otherwise(0)).alias("high_risk_members"),
    _sum(when(col("total_risk_score") < 1.0, 1).otherwise(0)).alias("low_risk_members")
).orderBy(col("total_projected_revenue").desc())

# Display results
print("📊 Revenue Forecast by Plan (PySpark):")
display(revenue_forecast_pyspark)

# Optionally save to table
# revenue_forecast_pyspark.write.format("delta").mode("overwrite").saveAsTable("payer_gold.revenue_forecast_pyspark")



## Example 3: HCC Distribution Analysis

### 📈 Business Goal
Understand which HCC categories drive the most revenue and identify coding opportunities.

This helps:
- Identify high-value diagnoses for provider education
- Monitor HCC capture rates
- Find gaps in documentation


### 📊 SAS vs. Databricks SQL Comparison

#### Traditional SAS Approach

```sas
/* SAS: HCC Distribution Analysis */
/* Step 1: Unnest HCC categories from member_risk_scores */
/* Note: SAS doesn't have native array explosion like SQL EXPLODE */
/* Must use DATA step with ARRAY processing */

DATA work.hcc_exploded;
    SET work.member_risk_scores;
    ARRAY hccs hcc_cat1-hcc_cat10;  /* Assumes max 10 HCCs per member */
    
    DO i = 1 TO DIM(hccs);
        IF hccs[i] NE . THEN DO;
            hcc_category = hccs[i];
            OUTPUT;
        END;
    END;
    DROP hcc_cat1-hcc_cat10 i;
RUN;

/* Step 2: Join with HCC reference and aggregate */
PROC SQL;
    CREATE TABLE work.hcc_distribution AS
    SELECT 
        he.hcc_category,
        r.diagnosis_desc,
        r.hcc_coefficient,
        COUNT(DISTINCT he.member_id) AS member_count,
        COUNT(DISTINCT he.plan_id) AS plan_count,
        ROUND(r.hcc_coefficient * COUNT(DISTINCT he.member_id) * 10000, 0.01) 
            AS total_revenue_impact,
        ROUND(r.hcc_coefficient * 10000, 0.01) AS revenue_per_member
    FROM work.hcc_exploded AS he
    INNER JOIN work.hcc_reference AS r 
        ON he.hcc_category = r.hcc_category
    GROUP BY he.hcc_category, r.diagnosis_desc, r.hcc_coefficient
    ORDER BY total_revenue_impact DESC;
QUIT;
```

**SAS Challenges:**
- ❌ No native EXPLODE function - requires manual array processing
- ❌ Must pre-define array size (max HCCs per member)
- ❌ Two-step process: DATA step + PROC SQL
- ❌ Complex logic for dynamic array sizes
- ❌ Performance issues with large datasets

#### Databricks SQL Approach - Single Statement!


## 🎯 YOUR TURN! HCC Distribution Analysis (5 mins)

### 📋 Business Context for Humana BI Team
As a BI analyst, you're asked to analyze **which HCC categories drive the most revenue** for Humana. This helps:
- 🎯 **Provider Education**: Focus on high-value HCC categories for better documentation
- 📈 **Revenue Optimization**: Identify opportunities for improved risk capture
- 📊 **Coding Gap Analysis**: Compare HCC capture rates across provider networks
- 💰 **Financial Planning**: Understand revenue composition by condition

### 🚫 The SAS Challenge: Array Processing

In SAS, this analysis is **complex** because you need to:
1. **Manually unnest arrays** (no EXPLODE function)
2. **Use DATA step + PROC SQL** (two separate steps)
3. **Pre-define array sizes** (what if a member has >10 HCCs?)

**Traditional SAS Approach (Complex!):**
```sas
/* SAS: HCC Distribution Analysis */
/* Step 1: Unnest HCC categories from member_risk_scores */
/* Note: SAS doesn't have native array explosion like SQL EXPLODE */
/* Must use DATA step with ARRAY processing */

DATA work.hcc_exploded;
    SET work.member_risk_scores;
    ARRAY hccs hcc_cat1-hcc_cat10;  /* Assumes max 10 HCCs per member */
    
    DO i = 1 TO DIM(hccs);
        IF hccs[i] NE . THEN DO;
            hcc_category = hccs[i];
            OUTPUT;
        END;
    END;
    DROP hcc_cat1-hcc_cat10 i;
RUN;

/* Step 2: Join with HCC reference and aggregate */
PROC SQL;
    CREATE TABLE work.hcc_distribution AS
    SELECT 
        he.hcc_category,
        r.diagnosis_desc,
        r.hcc_coefficient,
        COUNT(DISTINCT he.member_id) AS member_count,
        COUNT(DISTINCT he.plan_id) AS plan_count,
        ROUND(r.hcc_coefficient * COUNT(DISTINCT he.member_id) * 10000, 0.01) 
            AS total_revenue_impact,
        ROUND(r.hcc_coefficient * 10000, 0.01) AS revenue_per_member
    FROM work.hcc_exploded AS he
    INNER JOIN work.hcc_reference AS r 
        ON he.hcc_category = r.hcc_category
    GROUP BY he.hcc_category, r.diagnosis_desc, r.hcc_coefficient
    ORDER BY total_revenue_impact DESC;
QUIT;
```

### ⚡ Databricks Advantage: Native EXPLODE Function!

Databricks SQL has a native **`EXPLODE()`** function that makes this analysis much simpler:
- ✅ **Single SQL statement** (no DATA step needed)
- ✅ **No array size limits** (handles any number of HCCs)
- ✅ **Cleaner, more maintainable code**
- ✅ **Automatically distributed** across cluster

### 🔄 Your Task: Convert to Databricks SQL

Create a table called **`payer_gold.hcc_distribution`** that:
1. Explodes the `hcc_categories` array from `member_risk_scores`
2. Joins with `hcc_reference` to get descriptions and coefficients
3. Calculates member counts and revenue impact by HCC category
4. Orders by total revenue impact (descending)

### 💡 Key Hints:
- Use **`LATERAL VIEW EXPLODE(hcc_categories)`** to unnest the array
- Or use **`EXPLODE()`** in the SELECT clause with cross join
- The array column is named `hcc_categories` in `member_risk_scores`
- Join with `payer_gold.hcc_reference` on `hcc_category`

### 📝 Try it yourself in the next cell!

### 💡 Solution: HCC Distribution with EXPLODE

<details>
<summary>Click to reveal solution (try it yourself first!)</summary>

**Databricks SQL Solution:**
```sql
CREATE OR REPLACE TABLE payer_gold.hcc_distribution AS
SELECT
    hcc_category,
    r.diagnosis_desc,
    r.hcc_coefficient,
    COUNT(DISTINCT m.member_id) AS member_count,
    COUNT(DISTINCT m.plan_id) AS plan_count,
    ROUND(r.hcc_coefficient * COUNT(DISTINCT m.member_id) * 10000, 2) AS total_revenue_impact,
    ROUND(r.hcc_coefficient * 10000, 2) AS revenue_per_member
FROM payer_gold.member_risk_scores m
LATERAL VIEW EXPLODE(m.hcc_categories) AS hcc_category
INNER JOIN payer_gold.hcc_reference r ON hcc_category = r.hcc_category
GROUP BY hcc_category, r.diagnosis_desc, r.hcc_coefficient
ORDER BY total_revenue_impact DESC;
```

**Key Differences from SAS:**
- ✅ **Single SQL statement** vs. DATA step + PROC SQL
- ✅ **`LATERAL VIEW EXPLODE()`** replaces complex ARRAY processing
- ✅ **No array size limits** (SAS requires pre-defined hcc_cat1-hcc_cat10)
- ✅ **More readable and maintainable** code
- ✅ **Automatically optimized** by Spark's Catalyst optimizer

**Alternative Syntax (using inline EXPLODE):**
```sql
-- You can also use this syntax:
SELECT
    exploded.hcc_category,
    r.diagnosis_desc,
    r.hcc_coefficient,
    COUNT(DISTINCT m.member_id) AS member_count,
    COUNT(DISTINCT m.plan_id) AS plan_count,
    ROUND(r.hcc_coefficient * COUNT(DISTINCT m.member_id) * 10000, 2) AS total_revenue_impact
FROM payer_gold.member_risk_scores m,
LATERAL EXPLODE(m.hcc_categories) AS exploded(hcc_category)
INNER JOIN payer_gold.hcc_reference r ON exploded.hcc_category = r.hcc_category
GROUP BY exploded.hcc_category, r.diagnosis_desc, r.hcc_coefficient
ORDER BY total_revenue_impact DESC;
```

</details>


In [0]:
%sql
CREATE OR REPLACE TABLE payer_gold.hcc_distribution AS
SELECT
    exploded.hcc_category,
    r.diagnosis_desc,
    r.hcc_coefficient,
    COUNT(DISTINCT m.member_id) AS member_count,
    COUNT(DISTINCT m.plan_id) AS plan_count,
    ROUND(r.hcc_coefficient * COUNT(DISTINCT m.member_id) * 10000, 2) AS total_revenue_impact,
    ROUND(r.hcc_coefficient * 10000, 2) AS revenue_per_member
FROM payer_gold.member_risk_scores m
INNER JOIN payer_gold.hcc_reference r
    ON ARRAY_CONTAINS(m.hcc_categories, r.hcc_category)
LATERAL VIEW EXPLODE(m.hcc_categories) exploded AS hcc_category
GROUP BY exploded.hcc_category, r.diagnosis_desc, r.hcc_coefficient
ORDER BY total_revenue_impact DESC;

SELECT * FROM payer_gold.hcc_distribution LIMIT 20;

### 🐍 BONUS: PySpark Alternative with explode()

PySpark also makes array explosion simple:


In [0]:
# 🐍 PySpark Solution: HCC Distribution with explode()
from pyspark.sql.functions import (
    col, explode, countDistinct, count, round as spark_round
)

# Read tables
member_scores = spark.table("payer_gold.member_risk_scores")
hcc_reference = spark.table("payer_gold.hcc_reference")

# Explode the HCC categories array and join with reference
hcc_distribution_pyspark = member_scores \
    .select(
        "member_id",
        "plan_id",
        explode("hcc_categories").alias("hcc_category")
    ) \
    .join(hcc_reference, "hcc_category") \
    .groupBy(
        "hcc_category",
        "diagnosis_desc",
        "hcc_coefficient"
    ) \
    .agg(
        countDistinct("member_id").alias("member_count"),
        countDistinct("plan_id").alias("plan_count"),
        spark_round(
            col("hcc_coefficient") * countDistinct("member_id") * 10000, 2
        ).alias("total_revenue_impact"),
        spark_round(col("hcc_coefficient") * 10000, 2).alias("revenue_per_member")
    ) \
    .orderBy(col("total_revenue_impact").desc())

# Display top results
print("📊 HCC Distribution by Revenue Impact (PySpark):")
display(hcc_distribution_pyspark.limit(20))

# Optionally save to table
# hcc_distribution_pyspark.write.format("delta").mode("overwrite").saveAsTable("payer_gold.hcc_distribution_pyspark")



## Example 4: Data Quality & Compliance Audit

### ✅ Business Goal
Ensure encounter data meets CMS submission standards and identify data quality issues.

**CMS Requirements:**
- Valid diagnosis codes (ICD-10 format)
- Complete member demographics
- Valid provider NPIs
- Service dates within coverage period


### 📊 SAS vs. Databricks SQL Comparison

#### Traditional SAS Approach

```sas
/* SAS: Data Quality Audit - Requires Multiple Queries */

/* Query 1: Total Encounters */
PROC SQL;
    CREATE TABLE work.audit_total AS
    SELECT 
        'Total Encounters' AS metric,
        COUNT(*) AS record_count,
        . AS pct_of_total,
        'INFO' AS severity
    FROM work.claims;
QUIT;

/* Query 2: Encounters Missing Diagnosis */
PROC SQL;
    CREATE TABLE work.audit_missing_dx AS
    SELECT 
        'Encounters Missing Diagnosis' AS metric,
        COUNT(DISTINCT c.claim_id) AS record_count,
        CALCULATED record_count * 100.0 / 
            (SELECT COUNT(*) FROM work.claims) AS pct_of_total,
        'ERROR' AS severity
    FROM work.claims AS c
    LEFT JOIN work.diagnosis_raw AS d 
        ON c.claim_id = d.claim_id
    WHERE d.claim_id IS NULL;
QUIT;

/* Query 3: HCC Mapping Rate */
PROC SQL;
    CREATE TABLE work.audit_hcc_mapped AS
    SELECT 
        'Diagnoses Mapped to HCC' AS metric,
        COUNT(*) AS record_count,
        CALCULATED record_count * 100.0 / 
            (SELECT COUNT(*) FROM work.diagnosis_raw) AS pct_of_total,
        'INFO' AS severity
    FROM work.diagnosis_with_hcc
    WHERE is_hcc = 1;
QUIT;

/* Repeat for remaining checks... */

/* Combine all audit results */
DATA work.data_quality_audit;
    SET work.audit_total
        work.audit_missing_dx
        work.audit_hcc_mapped
        /* ... other audit tables ... */;
        
    /* Add status flag */
    IF severity = 'ERROR' AND record_count > 0 THEN status = 'FAIL';
    ELSE IF severity = 'WARNING' AND pct_of_total > 5 THEN status = 'REVIEW';
    ELSE status = 'PASS';
RUN;

/* Sort and display */
PROC SORT DATA=work.data_quality_audit;
    BY severity DESCENDING pct_of_total;
RUN;
```

**SAS Challenges:**
- ❌ Must create separate queries for each audit check
- ❌ Manual combination with DATA step
- ❌ Multiple intermediate tables clutter workspace
- ❌ Difficult to maintain as audit rules grow
- ❌ No UNION ALL equivalent in single PROC SQL
- ❌ Subquery limitations in SELECT clause

#### Databricks SQL Approach - Elegant & Maintainable!


In [0]:
%sql
-- Data Quality Audit for CMS Submission
CREATE OR REPLACE TABLE payer_gold.data_quality_audit AS
WITH encounter_checks AS (
  SELECT
    'Total Encounters' as metric,
    COUNT(*) as record_count,
    NULL as pct_of_total,
    'INFO' as severity
  FROM payer_silver.claims

  UNION ALL

  SELECT
    'Encounters Missing Diagnosis',
    COUNT(DISTINCT c.claim_id),
    ROUND(
      COUNT(DISTINCT c.claim_id) * 100.0 / (SELECT COUNT(*) FROM payer_silver.claims),
      2
    ),
    'ERROR'
  FROM payer_silver.claims c
  LEFT JOIN payer_bronze.diagnosis_raw d ON c.claim_id = d.claim_id
  WHERE d.claim_id IS NULL

  UNION ALL

  SELECT
    'Diagnoses Mapped to HCC',
    COUNT(*),
    ROUND(
      COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payer_bronze.diagnosis_raw),
      2
    ),
    'INFO'
  FROM payer_gold.diagnosis_with_hcc
  WHERE is_hcc = 1

  UNION ALL

  SELECT
    'Members Without Risk Scores',
    COUNT(DISTINCT m.member_id),
    ROUND(
      COUNT(DISTINCT m.member_id) * 100.0 / (SELECT COUNT(*) FROM payer_silver.members),
      2
    ),
    'WARNING'
  FROM payer_silver.members m
  LEFT JOIN payer_gold.member_risk_scores r ON m.member_id = r.member_id
  WHERE r.member_id IS NULL

  UNION ALL

  SELECT
    'Providers Missing NPI',
    COUNT(*),
    ROUND(
      COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payer_silver.providers),
      2
    ),
    'ERROR'
  FROM payer_silver.providers
  WHERE npi IS NULL OR TRIM(npi) = ''

  UNION ALL

  SELECT
    'Claims with Invalid Status',
    COUNT(*),
    ROUND(
      COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payer_silver.claims),
      2
    ),
    'WARNING'
  FROM payer_silver.claims
  WHERE claim_status NOT IN ('approved', 'paid', 'pending')
)
SELECT
  metric,
  record_count,
  COALESCE(pct_of_total, 0.0) as pct_of_total,
  severity,
  CASE
    WHEN severity = 'ERROR' AND record_count > 0 THEN 'FAIL'
    WHEN severity = 'WARNING' AND pct_of_total > 5 THEN 'REVIEW'
    ELSE 'PASS'
  END as status
FROM encounter_checks
ORDER BY
  CASE severity WHEN 'ERROR' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
  pct_of_total DESC;

In [0]:
%sql
-- View data quality audit results
SELECT * FROM payer_gold.data_quality_audit;


## Example 5: Member Risk Stratification

### 👥 Business Goal
Segment members by risk level to support care management and intervention programs.

**Risk Tiers:**
- **Very High Risk** (Score > 2.0): Intensive care management
- **High Risk** (Score 1.5-2.0): Enhanced monitoring
- **Moderate Risk** (Score 1.0-1.5): Standard care
- **Low Risk** (Score < 1.0): Preventive care focus


In [0]:
%sql
-- Member Risk Stratification
CREATE OR REPLACE TABLE payer_gold.member_risk_stratification AS
SELECT
  member_id,
  CONCAT(first_name, ' ', last_name) as member_name,
  age,
  gender,
  plan_id,
  total_risk_score,
  hcc_count,
  projected_annual_payment,
  CASE
    WHEN total_risk_score >= 2.0 THEN 'Very High Risk'
    WHEN total_risk_score >= 1.5 THEN 'High Risk'
    WHEN total_risk_score >= 1.0 THEN 'Moderate Risk'
    ELSE 'Low Risk'
  END as risk_tier,
  CASE
    WHEN total_risk_score >= 2.0 THEN 'Intensive Care Management Required'
    WHEN total_risk_score >= 1.5 THEN 'Enhanced Monitoring Recommended'
    WHEN total_risk_score >= 1.0 THEN 'Standard Care Protocol'
    ELSE 'Preventive Care Focus'
  END as care_recommendation,
  hcc_categories as active_hcc_list
FROM payer_gold.member_risk_scores;


In [0]:
%sql
-- Summary statistics by risk tier
SELECT
  risk_tier,
  COUNT(*) as member_count,
  ROUND(AVG(age), 1) as avg_age,
  ROUND(AVG(total_risk_score), 3) as avg_risk_score,
  ROUND(AVG(hcc_count), 1) as avg_hcc_count,
  SUM(projected_annual_payment) as total_revenue,
  ROUND(AVG(projected_annual_payment), 2) as avg_payment_per_member
FROM payer_gold.member_risk_stratification
GROUP BY risk_tier
ORDER BY avg_risk_score DESC;



## Example 6: Provider Performance on Risk Capture

### 🏥 Business Goal
Identify which providers excel at documenting HCC conditions to guide provider education.

**Key Metrics:**
- Members per provider
- Average risk score of provider's panel
- HCC capture rate
- Revenue attributed to provider


### 📊 SAS vs. PySpark Comparison

#### Traditional SAS Approach

```sas
/* SAS: Provider Performance on Risk Capture */
PROC SQL;
    CREATE TABLE work.provider_performance AS
    SELECT 
        p.provider_id,
        p.provider_name,
        p.specialty,
        p.city,
        p.state,
        COUNT(DISTINCT c.member_id) AS unique_members,
        COUNT(c.claim_id) AS total_encounters,
        ROUND(AVG(m.total_risk_score), 0.001) AS avg_member_risk_score,
        SUM(dh.hcc_coefficient) AS total_hcc_value,
        COUNT(DISTINCT dh.hcc_category) AS unique_hccs_captured,
        ROUND(SUM(m.projected_annual_payment), 0.01) AS attributed_revenue,
        ROUND(COUNT(DISTINCT dh.hcc_category) / COUNT(DISTINCT c.member_id), 0.01) 
            AS hcc_capture_rate
    FROM work.claims AS c
    INNER JOIN work.providers AS p 
        ON c.provider_id = p.provider_id
    INNER JOIN work.member_risk_scores AS m 
        ON c.member_id = m.member_id
    INNER JOIN work.diagnosis_with_hcc AS dh 
        ON c.claim_id = dh.claim_id
    GROUP BY p.provider_id, p.provider_name, p.specialty, p.city, p.state
    ORDER BY attributed_revenue DESC;
QUIT;

/* Create top providers report */
PROC PRINT DATA=work.provider_performance(OBS=20);
    TITLE 'Top 20 Providers by Risk Capture Performance';
    VAR provider_name specialty unique_members avg_member_risk_score 
        unique_hccs_captured attributed_revenue hcc_capture_rate;
RUN;
```

**SAS Limitations:**
- ❌ Must calculate derived metrics (hcc_capture_rate) in the same SELECT
- ❌ No withColumn() equivalent for cleaner syntax
- ❌ Limited to single-server memory for large joins
- ❌ Static output - no interactive display
- ❌ Requires separate PROC PRINT for visualization

#### PySpark Approach - More Flexible & Scalable!


In [0]:
from pyspark.sql.functions import count, countDistinct, avg, sum, round as spark_round, col

# Provider Performance Analysis
claims = spark.table("payer_silver.claims")
providers = spark.table("payer_silver.providers")
members = spark.table("payer_gold.member_risk_scores")
diagnosis_hcc = spark.table("payer_gold.diagnosis_with_hcc")

# Join claims with risk scores
provider_performance = claims \
    .join(providers, "provider_id") \
    .join(members, "member_id") \
    .join(diagnosis_hcc, "claim_id") \
    .groupBy(
        "provider_id",
        "provider_name",
        "specialty",
        "city",
        "state"
    ) \
    .agg(
        countDistinct("member_id").alias("unique_members"),
        count("claim_id").alias("total_encounters"),
        spark_round(avg("total_risk_score"), 3).alias("avg_member_risk_score"),
        sum(col("hcc_coefficient")).alias("total_hcc_value"),
        countDistinct(col("hcc_category")).alias("unique_hccs_captured"),
        spark_round(sum("projected_annual_payment"), 2).alias("attributed_revenue")
    ) \
    .withColumn(
        "hcc_capture_rate",
        spark_round(col("unique_hccs_captured") / col("unique_members"), 2)
    ) \
    .orderBy(col("attributed_revenue").desc())

# Display top providers
print("🏥 Top 20 Providers by Risk Capture Performance:")
display(provider_performance.limit(20))

# Save to Gold table
provider_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("payer_gold.provider_risk_capture_performance")



## Example 7: Encounter Datamart for CMS Submission

### 📊 Business Goal
Create a CMS-ready encounter datamart with all required fields and validations.

**CMS Submission Requirements:**
- Valid member enrollment
- Complete encounter details (dates, provider, diagnosis)
- Proper diagnosis code formatting (ICD-10)
- Service within coverage period
- Provider has valid NPI


In [0]:
%sql
-- Encounter Datamart for CMS Submission
CREATE OR REPLACE TABLE payer_gold.encounter_datamart_cms AS
SELECT
  -- Encounter identifiers
  c.claim_id as encounter_id,
  c.claim_date as encounter_date,
  c.claim_status as encounter_status,
  
  -- Member information
  m.member_id,
  m.first_name,
  m.last_name,
  m.birth_date,
  m.gender,
  m.plan_id,
  YEAR(CURRENT_DATE()) - YEAR(m.birth_date) as member_age,
  
  -- Provider information
  p.provider_id,
  p.npi as provider_npi,
  p.provider_name,
  p.specialty as provider_specialty,
  p.state as provider_state,
  
  -- Diagnosis information
  d.diagnosis_code as icd10_code,
  d.diagnosis_desc,
  d.hcc_category,
  d.hcc_coefficient,
  d.is_hcc,
  
  -- Claim financial
  c.total_charge,
  
  -- Data quality flags
  CASE 
    WHEN m.member_id IS NULL THEN 'FAIL: Missing Member'
    WHEN p.npi IS NULL OR TRIM(p.npi) = '' THEN 'FAIL: Missing Provider NPI'
    WHEN d.diagnosis_code IS NULL THEN 'FAIL: Missing Diagnosis'
    WHEN c.claim_date < m.effective_date THEN 'FAIL: Service Before Coverage'
    WHEN c.claim_status NOT IN ('approved', 'paid') THEN 'WARNING: Invalid Status'
    ELSE 'PASS'
  END as submission_validation_status,
  
  -- Submission flag
  CASE 
    WHEN m.member_id IS NOT NULL 
     AND p.npi IS NOT NULL 
     AND TRIM(p.npi) != ''
     AND d.diagnosis_code IS NOT NULL
     AND c.claim_date >= m.effective_date
     AND c.claim_status IN ('approved', 'paid')
    THEN 1 
    ELSE 0 
  END as cms_submission_ready,
  
  CURRENT_TIMESTAMP() as datamart_created_at
  
FROM payer_silver.claims c
INNER JOIN payer_silver.members m ON c.member_id = m.member_id
INNER JOIN payer_silver.providers p ON c.provider_id = p.provider_id
LEFT JOIN payer_gold.diagnosis_with_hcc d ON c.claim_id = d.claim_id;


In [0]:
%sql
-- CMS Submission Readiness Summary
SELECT
  submission_validation_status,
  COUNT(*) as encounter_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct_of_total,
  SUM(cms_submission_ready) as ready_for_submission
FROM payer_gold.encounter_datamart_cms
GROUP BY submission_validation_status
ORDER BY encounter_count DESC;


## 📊 Summary: SAS to Databricks Migration Guide for Humana BI Analysts

### 🎯 Key Takeaways for Humana Business Intelligence Team

As a BI analyst transitioning from SAS to Databricks, here are the most important concepts you've learned:

#### 1️⃣ **SQL Simplification**

| **SAS Pattern** | **Databricks SQL Equivalent** | **Benefit** |
|----------------|------------------------------|-------------|
| `PROC SQL` + `DATA Step` + `PROC SORT` | Single SQL query with CTEs | Cleaner, more maintainable |
| `CALCULATED keyword` for derived columns | Direct reference to derived columns | Natural SQL syntax |
| `PROC RANK` for ranking | `ROW_NUMBER() OVER()` window function | Built-in, no separate PROC |
| Manual array processing with `ARRAY` | `EXPLODE()` for arrays | Native support, no size limits |
| Multiple intermediate tables | CTEs (`WITH` clause) | No workspace clutter |
| `ROUND(value, 0.001)` | `ROUND(value, 3)` | Decimal places, not fractions |

#### 2️⃣ **Advanced Features Not Available in SAS**

✅ **`LATERAL VIEW EXPLODE()`** - Unnest arrays in single query (no DATA step)  
✅ **`COLLECT_SET()`** - Aggregate values into arrays  
✅ **`QUALIFY` clause** - Filter window function results elegantly  
✅ **Delta Lake** - ACID transactions, time travel, schema evolution  
✅ **Unity Catalog** - Built-in governance and lineage tracking  
✅ **Auto-scaling** - Distributed processing handles billions of rows  

#### 3️⃣ **Common Translation Patterns**

**SAS to Databricks SQL Quick Reference:**

```sql
/* SAS */                              /* Databricks SQL */
PROC SQL;                              -- No PROC needed, just SQL
CREATE TABLE work.table_name AS       CREATE OR REPLACE TABLE catalog.schema.table_name AS
UPCASE(column)                         UPPER(column)
CALCULATED derived_col                 derived_col (direct reference)
ROUND(value, 0.01)                     ROUND(value, 2)
COUNT(DISTINCT col)                    COUNT(DISTINCT col) -- same!
CASE WHEN ... END                      CASE WHEN ... END -- same!
QUIT;                                  -- No QUIT needed
```

**Window Functions:**
```sql
/* SAS - Limited OVER support */      /* Databricks SQL - Full Support */
SUM(sales) OVER (PARTITION BY...)     SUM(sales) OVER (PARTITION BY...)
                                       ROW_NUMBER() OVER (PARTITION BY...)
                                       RANK() OVER (PARTITION BY...)
                                       DENSE_RANK() OVER (PARTITION BY...)
                                       LAG() / LEAD() OVER (...)
```

#### 4️⃣ **Production Best Practices**

For **reliable, deterministic pipelines** at Humana:

1. ✅ **Write Gold tables to Delta** (not just views) for deterministic results
2. ✅ **Use CTEs** for complex logic instead of temp tables
3. ✅ **Apply partitioning** on date columns for query performance
4. ✅ **Enable time travel** for audit compliance (`SELECT * FROM table VERSION AS OF 10`)
5. ✅ **Use Unity Catalog** for data governance and PHI/PII protection
6. ✅ **Create views** for frequently-changing business logic
7. ✅ **Document lineage** automatically tracked by Unity Catalog

#### 5️⃣ **When to Use SQL vs. PySpark**

| **Use Databricks SQL When...** | **Use PySpark When...** |
|--------------------------------|-------------------------|
| Standard aggregations and joins | Complex business logic with conditionals |
| Reports and dashboards | Machine learning or advanced analytics |
| Data quality audits | Custom UDFs or complex transformations |
| BI analyst-friendly queries | Integration with Python libraries (pandas, numpy) |
| Quick ad-hoc analysis | Large-scale ETL with checkpointing |

Both are equally powerful! Choose based on team skills and use case.

---


# AI/BI

Intelligent analytics for everyone!

Databricks AI/BI is a new type of business intelligence product designed to provide a deep understanding of your data's semantics, enabling self-service data analysis for everyone in your organization. AI/BI is built on a compound AI system that draws insights from the full lifecycle of your data across the Databricks platform, including ETL pipelines, lineage, and other queries.

<img src="https://www.databricks.com/sites/default/files/2025-05/hero-image-ai-bi-v2-2x.png?v=1748417271" alt="Managed Tables" width="600" height="500">

# Genie

Talk with your data

Now everyone can get insights from data simply by asking questions in natural language.

<img src="https://www.databricks.com/sites/default/files/2025-06/ai-bi-genie-hero.png?v=1749162682" alt="Managed Tables" width="600" height="500">


# 🎓 Workshop Summary & Next Steps

## 🎉 Congratulations, Humana BI Team!

You've completed the **Databricks Humana Data & Analytics Workshop** specifically designed for Humana's Business Intelligence and Data Analytics team! Let's review what you learned:

---

## 🚀 Next Steps for Your Databricks Journey

### 📚 **Continue Learning**
1. **Databricks SQL Analytics** - Build interactive dashboards and reports
2. **Advanced Window Functions** - Moving averages, cumulative sums, percentile ranks
3. **Delta Live Tables** - Declarative ETL pipelines with data quality constraints
4. **Databricks Workflows** - Schedule and orchestrate production pipelines
5. **Machine Learning** - Predictive models for risk score forecasting

### 🛠️ **Apply to Your Work**
1. **Start Small**: Convert one SAS report to Databricks SQL
2. **Build Gold Tables**: Create analytics tables for your team's use cases
3. **Share Notebooks**: Collaborate with colleagues on Databricks
4. **Create Dashboards**: Use Databricks SQL Dashboards for executive reporting
5. **Automate Pipelines**: Schedule recurring jobs with Workflows

### 📖 **Resources**
- [Databricks SQL Reference](https://docs.databricks.com/sql/language-manual/index.html)
- [PySpark API Documentation](https://spark.apache.org/docs/latest/api/python/)
- [Delta Lake Guide](https://docs.databricks.com/delta/index.html)
- [Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/index.html)
- **Best Practices Notebook**: _[Reference] Best Practices_

### 💡 **Tips for Success**
- ✅ **Use AI Assistant** - Ask questions, get code suggestions
- ✅ **Read Documentation** - Databricks has excellent docs
- ✅ **Experiment** - Try different approaches, optimize queries
- ✅ **Collaborate** - Share notebooks, learn from peers
- ✅ **Think in SQL** - Most SAS PROC SQL translates directly!

---

## 📝 Feedback

We'd love to hear your thoughts on this workshop!

**What worked well?** What could be improved? **What SAS-to-Databricks topics do you want to learn next?**

Share feedback with your training coordinator or team lead.

---

## 🙏 Thank You!

Thank you for participating in this workshop designed specifically for **Humana's Business Intelligence and Data Analytics team**. 

We hope you found it valuable and are excited to continue your Databricks journey! 🚀

**Welcome to the future of healthcare analytics at Humana!** 💙

---
